# Phase 2-3: Data Preprocessing & DistilBERT Sentiment+Embeddings

This notebook performs three phases:
- **Phase 1**: Load raw data (ratings, reviews, metadata)
- **Phase 2**: Clean, encode, and create train/val/test splits (time-aware per user)
- **Phase 3**: Extract DistilBERT sentiment scores and embeddings from reviews

## Key Components:
- **Sentiment Analysis**: DistilBERT fine-tuned on SST-2 → -1 to +1 scores
- **Embeddings**: DistilBERT [CLS] token → 768-dim vectors (from same model)
- **Metadata**: Categories, Brands, Artists for Knowledge Graph construction
- **Outputs**: train/val/test CSVs, ID mappings, statistics, embeddings for KG + node2vec

**Run Time**: ~10-15 mins
**No 5-core filtering** - uses all data (836K interactions)

In [ ]:
!pip install -q transformers torch pandas numpy scikit-learn tqdm -q

In [ ]:
import os
import pickle
import pandas as pd
import numpy as np
import torch
from transformers import pipeline, AutoTokenizer, AutoModel
from sklearn.preprocessing import LabelEncoder
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

Device: cuda
GPU: Tesla T4
GPU Memory: 15.6 GB
Random seeds set.


## PHASE 2: Data Processing & Split

### Data Flow:
1. **Load**: Raw ratings + review text + metadata
2. **Clean**: Remove missing values, filter valid ratings (1-5 stars)
3. **Encode**: User/item IDs → 0-indexed
4. **Split**: Time-aware per-user split (70% train, 15% val, 15% test)
   - Each user's interactions chronologically ordered
   - Recent interactions → test/val to reflect deployment scenario

### Data Statistics:
Uses all available data (no 5-core filtering):
- ~5,500 unique users
- ~3,600 unique items  
- ~836K total interactions (ratings + reviews)
- Each user has min 1 interaction (includes cold-start scenarios)

## PHASE 3: DistilBERT Sentiment & Embeddings

### Sentiment Pipeline:
1. Load DistilBERT fine-tuned on SST-2
2. Extract sentiment from review text → [-1, +1] scale
3. Aggregate per user/item for rating bias terms

### Embedding Extraction:
1. **Model**: Same DistilBERT (distilbert-base-uncased-finetuned-sst-2-english)
2. **Method**: Extract [CLS] token hidden state → 768-dim vector
3. **Aggregation**: Average embeddings over reviews per item
4. **Output**: item_embeddings dict for KG initialization

In [65]:
# Cell 3: Load reviews and metadata
# Detect environment and set paths
import os
import ast

BASE_DIR = os.getcwd()

# Check if running on real Kaggle (files exist there) or locally
if os.path.exists('/kaggle/input/datasets/chandrimanandi/kg-dataset/reviews_Digital_Music.json'):
    # Running on real Kaggle with uploaded dataset
    REVIEWS_PATH = '/kaggle/input/datasets/chandrimanandi/kg-dataset/reviews_Digital_Music.json'
    METADATA_PATH = '/kaggle/input/datasets/chandrimanandi/kg-dataset/meta_Digital_Music.json'
    OUTPUT_DIR = '/kaggle/working'
elif os.path.exists(os.path.join(BASE_DIR, 'reviews_Digital_Music.json')):
    # Files available locally
    REVIEWS_PATH = os.path.join(BASE_DIR, 'reviews_Digital_Music.json')
    METADATA_PATH = os.path.join(BASE_DIR, 'meta_Digital_Music.json')
    OUTPUT_DIR = os.path.join(BASE_DIR, 'output')
else:
    # Default to local (will show file not found error if not available)
    REVIEWS_PATH = os.path.join(BASE_DIR, 'reviews_Digital_Music.json')
    METADATA_PATH = os.path.join(BASE_DIR, 'meta_Digital_Music.json')
    OUTPUT_DIR = os.path.join(BASE_DIR, 'output')

print(f"BASE DIR: {BASE_DIR}")
print(f"REVIEWS_PATH: {REVIEWS_PATH} (exists: {os.path.exists(REVIEWS_PATH)})")
print(f"METADATA_PATH: {METADATA_PATH} (exists: {os.path.exists(METADATA_PATH)})")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")

os.makedirs(OUTPUT_DIR, exist_ok=True)

def load_jsonl(path):
    """Load JSONL file (handles both JSON and Python dict format)."""
    records = []
    skipped = 0
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                # Try JSON first
                records.append(json.loads(line))
            except json.JSONDecodeError:
                try:
                    # If JSON fails, try Python literal eval (handles single quotes)
                    records.append(ast.literal_eval(line))
                except (ValueError, SyntaxError):
                    skipped += 1
    return pd.DataFrame(records), skipped

# Load reviews
print("\n=== Loading reviews ===")
df_reviews, skipped_reviews = load_jsonl(REVIEWS_PATH)
print(f"  {len(df_reviews):,} reviews loaded (skipped {skipped_reviews} malformed)")
print(f"  Columns: {df_reviews.columns.tolist()}")

# Load metadata
print("\n=== Loading product metadata ===")
try:
    df_metadata, skipped_metadata = load_jsonl(METADATA_PATH)
    print(f"  ✓ {len(df_metadata):,} products loaded (skipped {skipped_metadata} malformed)")
    if len(df_metadata) > 0:
        print(f"  Columns: {df_metadata.columns.tolist()}")
    has_metadata = True
except FileNotFoundError:
    print("  ✗ Metadata file not found")
    has_metadata = False
    df_metadata = pd.DataFrame()
except Exception as e:
    print(f"  ✗ Error: {e}")
    has_metadata = False
    df_metadata = pd.DataFrame()

# Rename review columns
df_reviews = df_reviews.rename(columns={
    "reviewerID": "user_id",
    "asin": "item_id",
    "overall": "rating",
    "reviewText": "review_text",
    "unixReviewTime": "timestamp",
})

# Keep required review columns
review_cols = ["user_id", "item_id", "rating", "review_text", "timestamp", "summary"]
review_cols = [c for c in review_cols if c in df_reviews.columns]
df_reviews = df_reviews[review_cols].copy()

# Parse numeric columns
df_reviews["rating"] = pd.to_numeric(df_reviews["rating"], errors="coerce")
df_reviews["timestamp"] = pd.to_numeric(df_reviews["timestamp"], errors="coerce")
df_reviews = df_reviews.dropna(subset=["rating", "timestamp"])

print(f"\n  Reviews shape after parsing: {df_reviews.shape}")

# If metadata loaded successfully, rename and merge
if has_metadata and len(df_metadata) > 0:
    # Extract metadata from product entries
    if "asin" in df_metadata.columns:
        df_metadata = df_metadata.rename(columns={"asin": "item_id"})
    
    print(f"\n  Available metadata columns: {df_metadata.columns.tolist()}")
    
    # Safely extract each metadata field
    df_metadata["title"] = df_metadata.get("title", "Unknown")
    if isinstance(df_metadata["title"], pd.Series):
        df_metadata["title"] = df_metadata["title"].fillna("Unknown").astype(str)
    
    df_metadata["brand"] = df_metadata.get("brand", "Unknown")
    if isinstance(df_metadata["brand"], pd.Series):
        df_metadata["brand"] = df_metadata["brand"].fillna("Unknown").astype(str)
    
    # Extract first/main category
    def extract_main_category(categories):
        """Extract main category from nested list structure."""
        if isinstance(categories, list) and len(categories) > 0:
            if isinstance(categories[0], list) and len(categories[0]) > 0:
                # Format: [["Category1", "Category2", "Category3"], ...]
                return categories[0][0]  # Get first category in first list
            else:
                return categories[0]  # Direct list
        return "Unknown"
    
    if "categories" in df_metadata.columns:
        df_metadata["category"] = df_metadata["categories"].apply(extract_main_category)
    else:
        df_metadata["category"] = "Unknown"
    
    # Extract artist from title (common format: "Artist - Album")
    def extract_artist(title):
        """Extract artist name from title."""
        title_str = str(title) if pd.notna(title) else ""
        if title_str and "-" in title_str:
            artist = title_str.split("-")[0].strip()
            return artist if len(artist) > 0 else "Unknown"
        return "Unknown"
    
    df_metadata["artist"] = df_metadata["title"].apply(extract_artist)
    
    # Merge - only select columns that exist
    metadata_cols = ["item_id", "title", "brand", "category", "artist"]
    available_cols = [c for c in metadata_cols if c in df_metadata.columns]
    print(f"  Merging columns: {available_cols}")
    
    if "item_id" in df_metadata.columns:
        df_metadata_subset = df_metadata[available_cols].copy()
        df = df_reviews.merge(df_metadata_subset, on="item_id", how="left")
        
        # Fill missing metadata columns that weren't in the merge
        for col in ["title", "brand", "category", "artist"]:
            if col not in df.columns:
                df[col] = "Unknown"
            else:
                df[col] = df[col].fillna("Unknown").astype(str)
        
        print(f"  ✓ Metadata merged. Shape: {df.shape}")
    else:
        print("  ✗ 'item_id' column not found in metadata")
        df = df_reviews.copy()
        df["title"] = "Unknown"
        df["brand"] = "Unknown"
        df["category"] = "Unknown"
        df["artist"] = "Unknown"
else:
    print("  ✗ No metadata available - using reviews only")
    df = df_reviews.copy()
    df["title"] = "Unknown"
    df["brand"] = "Unknown"
    df["category"] = "Unknown"
    df["artist"] = "Unknown"

print(f"\n=== Metadata extraction results ===")
print(f"  Unique titles: {df['title'].nunique():,}")
print(f"  Unique brands: {df['brand'].nunique():,}")
print(f"  Unique categories: {df['category'].nunique():,}")
print(f"  Unique artists: {df['artist'].nunique():,}")

print(f"\nDate range: {pd.to_datetime(df['timestamp'], unit='s').min()} to {pd.to_datetime(df['timestamp'], unit='s').max()}")

BASE DIR: /kaggle/working
REVIEWS_PATH: /kaggle/input/datasets/chandrimanandi/kg-dataset/reviews_Digital_Music.json (exists: True)
METADATA_PATH: /kaggle/input/datasets/chandrimanandi/kg-dataset/meta_Digital_Music.json (exists: True)
OUTPUT_DIR: /kaggle/working

=== Loading reviews ===
  836,006 reviews loaded (skipped 0 malformed)
  Columns: ['reviewerID', 'asin', 'reviewerName', 'helpful', 'reviewText', 'overall', 'summary', 'unixReviewTime', 'reviewTime']

=== Loading product metadata ===
  ✓ 279,899 products loaded (skipped 0 malformed)
  Columns: ['asin', 'title', 'price', 'imUrl', 'related', 'salesRank', 'categories', 'description', 'brand']

  Reviews shape after parsing: (836006, 6)

  Available metadata columns: ['item_id', 'title', 'price', 'imUrl', 'related', 'salesRank', 'categories', 'description', 'brand']
  Merging columns: ['item_id', 'title', 'brand', 'category', 'artist']
  ✓ Metadata merged. Shape: (836006, 10)

=== Metadata extraction results ===
  Unique titles: 6,

In [67]:
# Cell 5: ID encoding
print("\n=== ID ENCODING ===")
user2idx = {u: i for i, u in enumerate(sorted(df["user_id"].unique()))}
item2idx = {it: i for i, it in enumerate(sorted(df["item_id"].unique()))}

df["user_idx"] = df["user_id"].map(user2idx)
df["item_idx"] = df["item_id"].map(item2idx)

n_users = len(user2idx)
n_items = len(item2idx)
print(f"Users: {n_users:,} | Items: {n_items:,}")
print(f"Sparsity: {1 - len(df)/(n_users*n_items):.4f}")


=== ID ENCODING ===
Users: 5,541 | Items: 3,568
Sparsity: 0.9967


In [68]:
# Cell 6: Sort by timestamp & time-aware split
print("\n=== TIME-AWARE SPLIT ===")
df = df.sort_values("timestamp").reset_index(drop=True)

def time_aware_split(df):
    """Per-user: last interaction → test, 2nd last → val, rest → train."""
    train_rows, val_rows, test_rows = [], [], []
    
    for _, group in df.groupby("user_idx"):
        group = group.sort_values("timestamp")
        if len(group) < 3:
            train_rows.extend(group.to_dict("records"))
        else:
            test_rows.append(group.iloc[-1].to_dict())
            val_rows.append(group.iloc[-2].to_dict())
            train_rows.extend(group.iloc[:-2].to_dict("records"))
    
    return pd.DataFrame(train_rows), pd.DataFrame(val_rows), pd.DataFrame(test_rows)

train_df, val_df, test_df = time_aware_split(df)
print(f"Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")
print(f"Total: {len(train_df) + len(val_df) + len(test_df):,}")


=== TIME-AWARE SPLIT ===
Train: 53,624 | Val: 5,541 | Test: 5,541
Total: 64,706


In [69]:
# Cell 7: Compute global statistics
print("\n=== GLOBAL STATISTICS ===")
global_mean = train_df["rating"].mean()
user_mean = train_df.groupby("user_idx")["rating"].mean().to_dict()
item_mean = train_df.groupby("item_idx")["rating"].mean().to_dict()

print(f"Global mean: {global_mean:.4f}")
print(f"User means: μ={np.mean(list(user_mean.values())):.4f}, σ={np.std(list(user_mean.values())):.4f}")
print(f"Item means: μ={np.mean(list(item_mean.values())):.4f}, σ={np.std(list(item_mean.values())):.4f}")


=== GLOBAL STATISTICS ===
Global mean: 4.2210
User means: μ=4.2533, σ=0.6953
Item means: μ=4.2371, σ=0.5461


## PHASE 3: BERT Sentiment Analysis

Text processing pipeline:
1. **BERT Sentiment**: DistilBERT extracts sentiment scores (-1 to +1) from review text
2. **Aggregation**: Average sentiment per item/user for rating prediction


In [70]:
# Cell 8: Metadata summary statistics
print("\n=== METADATA SUMMARY ===")

metadata_stats = {}
for col in ['title', 'brand', 'category', 'artist']:
    if col in df.columns:
        n_unique = df[col].nunique()
        top_values = df[col].value_counts().head(5)
        metadata_stats[col] = {
            'unique': n_unique,
            'top': top_values.to_dict()
        }
        print(f"\n{col.upper()}:")
        print(f"  Unique values: {n_unique}")
        print(f"  Top 5:")
        for val, count in top_values.items():
            print(f"    {val}: {count:,}")

# These will be used in KG construction
print(f"\n✓ Metadata ready for Knowledge Graph construction")
print(f"  Categories for item-category edges: {metadata_stats['category']['unique']}")
print(f"  Brands for item-brand edges: {metadata_stats['brand']['unique']}")
print(f"  Artists for item-artist edges: {metadata_stats['artist']['unique']}")



=== METADATA SUMMARY ===

TITLE:
  Unique values: 2629
  Top 5:
    Unknown: 10,867
    The Massacre: 272
    Get Rich Or Die Tryin: 271
    The Eminem Show [Limited Edition w/ Bonus DVD]: 204
    The Marshall Mathers LP: 202

BRAND:
  Unique values: 224
  Top 5:
    Unknown: 40,851
    : 10,238
    MUSIC: 868
    Umgd/Geffen: 632
    Sony: 562

CATEGORY:
  Unique values: 3
  Top 5:
    CDs & Vinyl: 61,495
    Digital Music: 3,203
    Arts, Crafts & Sewing: 8

ARTIST:
  Unique values: 104
  Top 5:
    Unknown: 62,705
    Elton John: 134
    Wu: 87
    De: 71
    So: 66

✓ Metadata ready for Knowledge Graph construction
  Categories for item-category edges: 3
  Brands for item-brand edges: 224
  Artists for item-artist edges: 104


In [71]:
# Cell 9: BERT Sentiment Extraction
print("\n=== BERT SENTIMENT ===")

# Load sentiment pipeline (device automatically set to GPU if available)
sentiment_pipe = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    device=0 if torch.cuda.is_available() else -1,
    truncation=True,
    max_length=512
)

print(f"Sentiment pipeline loaded on {sentiment_pipe.device}")

def extract_sentiment(texts, batch_size=64):
    """Extract sentiment scores via DistilBERT."""
    if isinstance(texts, pd.Series):
        texts = texts.tolist()
    
    results = sentiment_pipe(texts, batch_size=batch_size, truncation=True)
    scores = []
    for r in results:
        score = r["score"]
        if r["label"] == "NEGATIVE":
            score = -score
        scores.append(score)
    return scores

# Extract sentiment for all splits (using review_text column)
print("Extracting sentiment scores...")
train_df["sentiment"] = extract_sentiment(train_df["review_text"].tolist(), batch_size=128)
val_df["sentiment"]   = extract_sentiment(val_df["review_text"].tolist(),   batch_size=128)
test_df["sentiment"]  = extract_sentiment(test_df["review_text"].tolist(),  batch_size=128)

print(f"Sentiment range: [{train_df['sentiment'].min():.3f}, {train_df['sentiment'].max():.3f}]")


=== BERT SENTIMENT ===


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Sentiment pipeline loaded on cuda:0
Extracting sentiment scores...
Sentiment range: [-1.000, 1.000]


In [ ]:
# Cell 10: Extract DistilBERT Embeddings (from same model as sentiment)
print("\n=== DISTILBERT EMBEDDINGS ===")

# Use the same DistilBERT model for embeddings (768-dim: better than ST's 384)
from transformers import AutoTokenizer, AutoModel

model_name = "distilbert-base-uncased-finetuned-sst-2-english"
tokenizer = AutoTokenizer.from_pretrained(model_name)
embedding_model = AutoModel.from_pretrained(model_name, output_hidden_states=True)
embedding_model = embedding_model.to(device)
embedding_model.eval()

# DistilBERT produces 768-dim embeddings
embedding_dim = 768
print(f"DistilBERT embeddings: {embedding_dim}-dim (from [CLS] token)")

def extract_embeddings(texts, batch_size=32):
    """Extract DistilBERT embeddings from text."""
    embeddings = []
    
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i+batch_size]
        
        # Tokenize
        inputs = tokenizer(batch_texts, return_tensors="pt", truncation=True, 
                          max_length=512, padding=True)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        
        # Get embeddings
        with torch.no_grad():
            outputs = embedding_model(**inputs)
            # Use [CLS] token embedding (first token, last hidden state)
            batch_emb = outputs.last_hidden_state[:, 0, :].cpu().numpy()
        
        embeddings.extend(batch_emb)
    
    return np.array(embeddings)

print("Extracting DistilBERT embeddings for all splits...")


=== SENTENCE TRANSFORMER ===
Loading ST model: sentence-transformers/all-MiniLM-L6-v2...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding dimension: 384


In [75]:
# Cell 13: Compute sentiment statistics
print("\n=== SENTIMENT STATISTICS ===")

user_sent_mean = train_df.groupby("user_idx")["sentiment"].mean().to_dict()
item_sent_mean = train_df.groupby("item_idx")["sentiment"].mean().to_dict()

print(f"User sentiment: μ={np.mean(list(user_sent_mean.values())):.4f}, σ={np.std(list(user_sent_mean.values())):.4f}")
print(f"Item sentiment: μ={np.mean(list(item_sent_mean.values())):.4f}, σ={np.std(list(item_sent_mean.values())):.4f}")


=== SENTIMENT STATISTICS ===
User sentiment: μ=0.5955, σ=0.4460
Item sentiment: μ=0.6238, σ=0.3513


In [ ]:
# Cell 11b: Extract text embeddings for each item (from reviews)
print("Extracting item embeddings (aggregating reviews per item)...")

# Group reviews by item
item_reviews = df.groupby("item_idx")["review_text"].apply(list).to_dict()

# Extract embeddings for each item (average of review embeddings)
item_embeddings = {}
for item_idx in tqdm(range(n_items)):
    if item_idx in item_reviews:
        # Get all reviews for this item
        reviews = item_reviews[item_idx][:10]  # Use up to 10 reviews to limit compute
        
        # Extract embeddings
        embeddings = extract_embeddings(reviews)
        
        # Average embeddings for item
        item_embeddings[item_idx] = np.mean(embeddings, axis=0)
    else:
        # No reviews for this item - use random init
        item_embeddings[item_idx] = np.random.randn(embedding_dim) * 0.01

print(f"Extracted embeddings for {len(item_embeddings)} items")

In [ ]:
# Cell 14: Augment datasets with features
print("\n=== AUGMENTING DATASETS ===")

def augment_df(df, train_df, embedding_dict, name="train"):
    """Add mean rating/sentiment to each split."""
    df = df.copy()
    
    # Global means (from train only)
    global_mean_val = train_df["rating"].mean()
    user_means = train_df.groupby("user_idx")["rating"].mean().to_dict()
    item_means = train_df.groupby("item_idx")["rating"].mean().to_dict()
    user_sents = train_df.groupby("user_idx")["sentiment"].mean().to_dict()
    item_sents = train_df.groupby("item_idx")["sentiment"].mean().to_dict()
    
    df["user_rating_mean"]    = df["user_idx"].map(lambda u: user_means.get(u, global_mean_val))
    df["item_rating_mean"]    = df["item_idx"].map(lambda i: item_means.get(i, global_mean_val))
    df["user_sentiment_mean"] = df["user_idx"].map(lambda u: user_sents.get(u, 0.0)).fillna(0.0)
    df["item_sentiment_mean"] = df["item_idx"].map(lambda i: item_sents.get(i, 0.0)).fillna(0.0)
    
    return df

train_df = augment_df(train_df, train_df, None, "train")
val_df = augment_df(val_df, train_df, None, "val")
test_df = augment_df(test_df, train_df, None, "test")

print("Datasets augmented with statistical features.")


=== AUGMENTING DATASETS ===
Datasets augmented with statistical features.


In [ ]:
# Cell 13: Save all outputs
print("\n=== SAVING OUTPUTS ===")

# Save processed dataframes
train_df.to_csv(f"{OUTPUT_DIR}/train.csv", index=False)
val_df.to_csv(f"{OUTPUT_DIR}/val.csv", index=False)
test_df.to_csv(f"{OUTPUT_DIR}/test.csv", index=False)
print("✓ CSV files saved")

# Save ID mappings
id_maps = {
    'user2idx': user2idx,
    'idx2user': {v: k for k, v in user2idx.items()},
    'item2idx': item2idx,
    'idx2item': {v: k for k, v in item2idx.items()},
    'n_users': n_users,
    'n_items': n_items,
}
with open(f"{OUTPUT_DIR}/id_maps.pkl", "wb") as f:
    pickle.dump(id_maps, f)
print("✓ ID maps saved")

# Save statistics
with open(f"{OUTPUT_DIR}/rating_stats.pkl", "wb") as f:
    pickle.dump({
        'global_mean': global_mean,
        'user_mean': user_mean,
        'item_mean': item_mean,
    }, f)
print("✓ Rating stats saved")

# Save sentiment statistics
with open(f"{OUTPUT_DIR}/sentiment_stats.pkl", "wb") as f:
    pickle.dump({
        'user_sent_mean': user_sent_mean,
        'item_sent_mean': item_sent_mean,
    }, f)
print("✓ Sentiment stats saved")

# Save metadata mappings for Knowledge Graph
try:
    with open(f"{OUTPUT_DIR}/metadata_mappings.pkl", "wb") as f:
        pickle.dump(metadata_mappings, f)
    print("✓ Metadata mappings saved for KG construction")
except NameError:
    print("⚠ Metadata mappings not available (check if metadata extraction was run)")

# Save DistilBERT embeddings for Knowledge Graph initialization
with open(f"{OUTPUT_DIR}/embeddings.pkl", "wb") as f:
    pickle.dump({
        'item_embeddings': item_embeddings,
        'embedding_dim': embedding_dim,
    }, f)
print("✓ DistilBERT embeddings saved (768-dim from [CLS] token)")

# Summary
print(f"\n{'='*60}")
print(f"PHASE 2-3 COMPLETE")
print(f"{'='*60}")
print(f"Users: {n_users:,}")
print(f"Items: {n_items:,}")
print(f"Train samples: {len(train_df):,}")
print(f"Val samples: {len(val_df):,}")
print(f"Test samples: {len(test_df):,}")
print(f"\nAll files saved to: {OUTPUT_DIR}")
print(f"Ready for download and next phase!")



=== SAVING OUTPUTS ===
✓ CSV files saved
✓ ID maps saved
✓ Rating stats saved
✓ Sentiment stats saved
✓ Embeddings saved
✓ Metadata mappings saved for KG construction

PHASE 2-3 COMPLETE
Users: 5,541
Items: 3,568
Train samples: 53,624
Val samples: 5,541
Test samples: 5,541
Embedding dimension: 384

All files saved to: /kaggle/working
Ready for download and next phase!


In [78]:
# Cell 15b: Extract and save metadata mappings for KG
print("\n=== METADATA MAPPINGS FOR KNOWLEDGE GRAPH ===")

# Create metadata mappings from actual data
metadata_mappings = {
    'item_to_category': {},
    'item_to_brand': {},
    'item_to_artist': {},
    'item_to_title': {},
    'categories': [],
    'brands': [],
    'artists': [],
}

# Build mappings from training data (using actual metadata from dataset)
for _, row in train_df.iterrows():
    item_idx = int(row['item_idx'])
    
    # Category (from product metadata)
    category = str(row['category']).strip() if pd.notna(row['category']) else "Unknown"
    metadata_mappings['item_to_category'][item_idx] = category
    if category not in metadata_mappings['categories']:
        metadata_mappings['categories'].append(category)
    
    # Brand (from product metadata)
    brand = str(row['brand']).strip() if pd.notna(row['brand']) else "Unknown"
    metadata_mappings['item_to_brand'][item_idx] = brand
    if brand not in metadata_mappings['brands']:
        metadata_mappings['brands'].append(brand)
    
    # Artist (extracted from title)
    artist = str(row['artist']).strip() if pd.notna(row['artist']) else "Unknown"
    metadata_mappings['item_to_artist'][item_idx] = artist
    if artist not in metadata_mappings['artists']:
        metadata_mappings['artists'].append(artist)
    
    # Title (for reference)
    title = str(row['title']).strip() if pd.notna(row['title']) else "Unknown"
    metadata_mappings['item_to_title'][item_idx] = title

print(f"✓ Metadata extraction complete!")
print(f"\nUnique metadata per type:")
print(f"  Categories: {len(metadata_mappings['categories'])}")
print(f"  Brands: {len(metadata_mappings['brands'])}")
print(f"  Artists: {len(metadata_mappings['artists'])}")

print(f"\nExample metadata for item 0:")
if 0 in metadata_mappings['item_to_category']:
    print(f"  Title: {metadata_mappings['item_to_title'].get(0, 'Unknown')}")
    print(f"  Category: {metadata_mappings['item_to_category'][0]}")
    print(f"  Brand: {metadata_mappings['item_to_brand'][0]}")
    print(f"  Artist: {metadata_mappings['item_to_artist'][0]}")

print(f"\nTop 3 of each metadata type:")
print(f"  Categories: {sorted(metadata_mappings['categories'])[:3]}")
print(f"  Brands: {sorted(metadata_mappings['brands'])[:3]}")
print(f"  Artists: {sorted(metadata_mappings['artists'])[:3]}")



=== METADATA MAPPINGS FOR KNOWLEDGE GRAPH ===
✓ Metadata extraction complete!

Unique metadata per type:
  Categories: 3
  Brands: 224
  Artists: 104

Example metadata for item 0:
  Title: Memory of Trees
  Category: CDs & Vinyl
  Brand: Unknown
  Artist: Unknown

Top 3 of each metadata type:
  Categories: ['Arts, Crafts & Sewing', 'CDs & Vinyl', 'Digital Music']
  Brands: ['', '!iT Jeans', 'A-HA']
  Artists: ['11', '12', '30 Greatest Hits']
